In [ ]:
from sqlmodel import SQLModel, Field, create_engine, Relationship, Field
from pydantic import ConfigDict

class Item(SQLModel, table=True):
    __table_args__ = {'extend_existing': True}
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(default="", max_length=255, schema_extra={"icon": "house"})
    price: float
    image_url: str | None = None
    is_offer: bool | None = None
    # lines: list['OrderLine'] = Relationship(back_populates="item")
    
class Order(SQLModel, table=True):
    __table_args__ = {'extend_existing': True}
    id: int | None = Field(default=None, primary_key=True)
    # lines: list['OrderLine'] = Relationship(back_populates="order")

class OrderLine(SQLModel, table=True):
    __table_args__ = {'extend_existing': True}
    id: int | None = Field(default=None, primary_key=True)
    order_id: int = Field(foreign_key="order.id")
    item_id: int = Field(foreign_key="item.id")
    quantity: int
    # order: Order = Relationship(back_populates="lines")
    # item: Item = Relationship(back_populates="lines")

engine = create_engine("sqlite:///database.db")

def create_db_and_tables():
    SQLModel.metadata.create_all(engine)

create_db_and_tables()


TypeError: Field() got an unexpected keyword argument 'json_schema_extra'

In [72]:
from sqlmodel import Session, select
apple = Item(name="Apple", price=10.0, is_offer=True, image_url="https://example.com/test.jpg")
banana = Item(name="Banana", price=10.0, is_offer=True, image_url="https://example.com/test.jpg")
cherry = Item(name="Cherry", price=10.0, is_offer=True, image_url="https://example.com/test.jpg")
# with Session(engine) as session:
#     session.add(apple)
#     session.add(banana)
#     session.add(cherry)
#     session.commit()

result = None
with Session(engine) as session:
    result = session.exec(select(Item).order_by(Item.id))
    for item in result.all():
        print(item.id, item.name)


1 Apple
2 Banana
3 Cherry


In [76]:
props = apple.model_json_schema()
for k,v in props.get("properties").items():
    print(v['title'])
props

Id
Name
Price
Image Url
Is Offer


{'properties': {'id': {'anyOf': [{'type': 'integer'}, {'type': 'null'}],
   'default': None,
   'title': 'Id'},
  'name': {'default': '', 'maxLength': 255, 'title': 'Name', 'type': 'string'},
  'price': {'title': 'Price', 'type': 'number'},
  'image_url': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Image Url'},
  'is_offer': {'anyOf': [{'type': 'boolean'}, {'type': 'null'}],
   'default': None,
   'title': 'Is Offer'}},
 'required': ['price'],
 'title': 'Item',
 'type': 'object'}